# BMR legacy packing reference

This notebook is kept as the single dataset-specific legacy reference for multi-turn / mixed SFT packing. For new datasets, prefer the maintained `offline_packing/auto_pipe.sh` pipeline and the `current_pipeline_smoke_test/` example.

Use this notebook only when you need to inspect or reproduce the older BMR-specific Step 2 flow: token-length bins -> packed raw samples -> WebDataset conversion.

## Expected inputs

Run `s1_get_tokenlens_v4-sft.py` first. It should produce a token-info file with one record per line:

```text
sample_name:token_count
another_sample:1234
```

The notebook writes bins to `s2_ckpt/bins_boxs_mr_sft_8k.pkl`, which is the file expected by `s2_prepare_rawsamples-bmr_sft_780k-8k-fast.py`.

In [ ]:
from pathlib import Path
import pickle
import subprocess
import sys

EXAMPLE_DIR = Path.cwd().resolve()
REPO_ROOT = EXAMPLE_DIR.parents[2]
OFFLINE_PACKING_DIR = REPO_ROOT / "offline_packing"

if str(OFFLINE_PACKING_DIR) not in sys.path:
    sys.path.insert(0, str(OFFLINE_PACKING_DIR))

from s3_bin_packing import BinPacker

TOKEN_INFO = EXAMPLE_DIR / "token_info_MR_sft_780k_8k.txt"
BIN_DIR = EXAMPLE_DIR / "s2_ckpt"
BIN_FILE = BIN_DIR / "bins_boxs_mr_sft_8k.pkl"

CAPACITY = 8192
MAX_SAMPLES_PER_BIN = 10
ALGORITHM = "bfd"  # bfd or ffd

CONVERTER_SCRIPT = EXAMPLE_DIR / "s2_prepare_rawsamples-bmr_sft_780k-8k-fast.py"
WDS_SCRIPT = EXAMPLE_DIR / "s3_test_bmr_sft_780k-8k.sh"

print(f"Example directory: {EXAMPLE_DIR}")
print(f"Token info file: {TOKEN_INFO}")
print(f"Bin output file: {BIN_FILE}")

## Build token-capacity bins

This cell uses the current repository implementation in `offline_packing/s3_bin_packing.py` instead of the removed legacy `hashbacket` helper. It keeps the old BMR output filename so the downstream legacy converter can still consume the bins.

In [ ]:
if not TOKEN_INFO.exists():
    raise FileNotFoundError(
        f"Token info file not found: {TOKEN_INFO}. Run s1_get_tokenlens_v4-sft.py first or update TOKEN_INFO."
    )

BIN_DIR.mkdir(parents=True, exist_ok=True)

packer = BinPacker(capacity=CAPACITY, max_samples_per_bin=MAX_SAMPLES_PER_BIN)
packer.parse_token_info(str(TOKEN_INFO))
distribution = packer.analyze_distribution()

if ALGORITHM == "bfd":
    bins = packer.pack_best_fit_decreasing()
elif ALGORITHM == "ffd":
    bins = packer.pack_first_fit_decreasing()
else:
    raise ValueError(f"Unsupported algorithm: {ALGORITHM}")

with BIN_FILE.open("wb") as f:
    pickle.dump(bins, f)

print(f"Saved {len(bins):,} bins to {BIN_FILE}")

In [ ]:
total_samples = sum(len(bin_items) for bin_items in bins)
total_tokens = sum(int(item["l"]) for bin_items in bins for item in bin_items)
used_ratios = [sum(int(item["l"]) for item in bin_items) / CAPACITY for bin_items in bins]

print(f"Packed samples: {total_samples:,}")
print(f"Packed bins: {len(bins):,}")
print(f"Average bin utilization: {sum(used_ratios) / len(used_ratios):.2%}")
print(f"Minimum bin utilization: {min(used_ratios):.2%}")
print(f"Maximum bin utilization: {max(used_ratios):.2%}")

## Convert bins into legacy BMR raw packed samples

The legacy converter still contains dataset-specific constants such as source image and JSON roots. Review `SRC_DIR_IMGS`, `SRC_DIR_JSONS`, and `f_toklens_originalsample` in `s2_prepare_rawsamples-bmr_sft_780k-8k-fast.py` before enabling the next cell.

In [ ]:
RUN_LEGACY_CONVERTER = False

if RUN_LEGACY_CONVERTER:
    subprocess.run([sys.executable, str(CONVERTER_SCRIPT)], check=True, cwd=EXAMPLE_DIR)
else:
    print("Set RUN_LEGACY_CONVERTER=True after updating dataset-specific paths in the converter script.")

## Convert raw packed samples to WebDataset

After the raw packed samples exist, use `s3_test_bmr_sft_780k-8k.sh` or call `offline_packing/s5_convert_to_webdataset.py --mode bmr_pack` directly. The shell script is kept as a legacy example and also contains paths that should be adjusted for your environment.

In [ ]:
RUN_WDS_CONVERSION = False

if RUN_WDS_CONVERSION:
    subprocess.run(["bash", str(WDS_SCRIPT)], check=True, cwd=EXAMPLE_DIR)
else:
    print("Set RUN_WDS_CONVERSION=True after updating IN_SAMPLE_DIR and OUT_WDS_DIR in the shell script.")